In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from DATA.stock_invest_function import *
from statsmodels.stats.diagnostic import acorr_ljungbox
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
import matplotlib
import matplotlib.pyplot as plt


matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

In [2]:
# DB 접속 정보 설정
db_info = {
    'user': 'stox7412',         # 예: 'root'
    'password': 'Apt106503!~', # 예: '1234'
    # 'host': '192.168.0.230',
    'host': 'hystox74.synology.me',         # 예: 'localhost' 또는 IP
    'port': '3307',              # 기본 포트는 보통 3306
    'database': 'investar'        # 예: 'trade_data'
}

In [15]:
def forecast_future_4q_with_sarima(df, date_col='date', value_col='endog_var', use_log=True):
    # 날짜 결측값 체크
    if df[date_col].isna().any():
        print(f"❌ 날짜에 결측치 존재 → 예측 불가: {df[date_col].isna().sum()}개")
        return pd.DataFrame({
            'date': pd.date_range(start=pd.Timestamp.today(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    # 시계열 설정
    ts = df.set_index(date_col)[value_col].asfreq('Q')

    if ts.isna().any():
        print("❌ 값에 결측치가 존재 → 보간 또는 제거 필요")
        return pd.DataFrame({
            'date': pd.date_range(start=ts.index[-1] + pd.offsets.QuarterEnd(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    if use_log:
        ts_transformed = np.log(ts)
    else:
        ts_transformed = ts

    # SARIMA 파라미터 탐색
    p = d = q = P = D = Q = [0, 1]
    s = 4
    param_combinations = list(product(p, d, q))
    seasonal_combinations = list(product(P, D, Q))
    total_combinations = list(product(param_combinations, seasonal_combinations))

    best_aic = np.inf
    best_model = None

    for (order, seasonal) in total_combinations:
        seasonal_order = (*seasonal, s)
        try:
            model = SARIMAX(ts_transformed, order=order, seasonal_order=seasonal_order)
            result = model.fit(disp=False)
            if result.aic < best_aic:
                best_aic = result.aic
                best_model = result
        except:
            continue

    if best_model is None:
        print("❌ 적합한 모델을 찾지 못했습니다.")
        return pd.DataFrame({
            'date': pd.date_range(start=ts.index[-1] + pd.offsets.QuarterEnd(), periods=4, freq='Q'),
            'revenue_forecast': [np.nan] * 4
        })

    # 예측 수행
    forecast_log = best_model.forecast(steps=4)
    forecast = np.exp(forecast_log) if use_log else forecast_log

    result_df = pd.DataFrame({
        'date': forecast.index,
        'revenue_forecast': forecast.values
    })

    return result_df

def forecast_multiple_symbols(filtered_df, date_col='date', value_col='value', min_periods=20):
    """
    매출이 0인 값을 제외한 뒤, 시계열 길이가 최소 min_periods 이상인 경우에만 SARIMA 예측 수행.

    Parameters:
        filtered_df (pd.DataFrame): 'date', 'ticker', 'value' 포함된 전체 데이터프레임
        date_col (str): 날짜 컬럼명
        value_col (str): 매출 등 예측 대상 값 컬럼명
        min_periods (int): 예측을 위한 최소 데이터 개수 (기본: 20개)

    Returns:
        Tuple[pd.DataFrame, list]:
            - 예측 결과 DataFrame
            - 예측 제외된 종목 리스트
    """
    result_list = []
    excluded_tickers = []

    for symbol in filtered_df['ticker'].unique():
        sub_df = filtered_df[filtered_df['ticker'] == symbol].copy()
        sub_df = sub_df[[date_col, value_col]].dropna()

        # ⚠️ 매출 0인 항목 제거
        sub_df = sub_df[sub_df[value_col] != 0]

        # ⚠️ 0 제외 후 길이 판단
        if len(sub_df) < min_periods:
            print(f"⛔ {symbol}: 매출 0 제외 후 유효 데이터 {len(sub_df)}개 → 예측 제외")
            excluded_tickers.append(symbol)
            continue

        sub_df = sub_df.rename(columns={value_col: 'endog_var'})

        forecast_df = forecast_future_4q_with_sarima(sub_df, date_col=date_col)

        forecast_df = forecast_df.rename(columns={
            'tic_name': 'ticker',
            'revenue_forecast': 'value'
        })
        forecast_df['ticker'] = symbol
        forecast_df['forecast'] = 1

        result_list.append(forecast_df)

    final_df = pd.concat(result_list, ignore_index=True) if result_list else pd.DataFrame(columns=['date', 'ticker', 'value', 'forecast'])

    print("📌 최종 컬럼 목록:", final_df.columns.tolist())
    print("🚫 예측 제외된 종목 수:", len(excluded_tickers))

    return final_df[['date', 'ticker', 'value', 'forecast']], excluded_tickers


In [16]:
# tic_name = "A000660"

fs_df = fetch_table_data(db_info, "korea_fs_data")
fs_df.rename(columns={'Date': 'date'}, inplace=True)

# 1. indicator 필터링
target_indicator = '매출액(천원)'
filtered_df = fs_df[fs_df['indicator'] == target_indicator].copy()

# 2. 날짜 정제 및 정렬
filtered_df['date'] = pd.to_datetime(filtered_df['date'])
filtered_df.sort_values(by='date', inplace=True)
filtered_df.rename(columns={'symbol': 'ticker'}, inplace=True)

# 3. value 컬럼이 있는지 확인 및 타입 강제
if 'value' not in filtered_df.columns:
    raise KeyError("'value' 컬럼이 없습니다.")

filtered_df['value'] = pd.to_numeric(filtered_df['value'], errors='coerce')

# 4. 피벗 테이블 생성 (행: date, 열: Symbol, 값: value)
pivot_df = filtered_df.pivot_table(
    index='date',
    columns='ticker',
    values='value',
    aggfunc='first'  # 중복 방지
)

# 5. 전년 동분기 대비 변화율 계산 (4분기 전 대비)
fs_yoy_growth_df = pivot_df.pct_change(periods=4) * 100

# endog_df = pivot_df[[tic_name]].reset_index()

✅ 'korea_fs_data' 테이블에서 5584577건의 데이터를 가져왔습니다.


In [18]:
def compute_yoy_and_ttm_growth(df):
    # 정렬
    df = df.sort_values(by=['ticker', 'date']).reset_index(drop=True)

    # YoY 성장률: 4분기 전 대비 비율
    df['yoy_growth'] = df.groupby('ticker')['value'].transform(lambda x: x.pct_change(periods=4))

    # TTM 값 계산: 최근 4개 분기의 평균
    df['ttm_current'] = df.groupby('ticker')['value'].transform(lambda x: x.rolling(window=4).mean())

    # 1년 전 TTM 평균
    df['ttm_past'] = df.groupby('ticker')['ttm_current'].shift(4)

    # TTM 성장률 계산
    df['TTM_growth'] = (df['ttm_current'] - df['ttm_past']) / df['ttm_past'].abs()

    # 불필요한 중간 컬럼 제거
    # df = df.drop(columns=['ttm_current', 'ttm_past'])

    return df


In [19]:
tic_list = filtered_df['ticker'].unique().tolist()

In [50]:
sorted_df = filtered_df[filtered_df['ticker'].isin(tic_list)].sort_values(by=['ticker','date'])
sorted_df_resize = sorted_df[['date', 'ticker', 'value']]
sorted_df_resize['forecast'] = 0
sales_data_df = sorted_df_resize[(sorted_df_resize['value']!=0)]
forecast_df, excluded = forecast_multiple_symbols(sales_data_df)
print("제외된 종목 목록:", excluded)

❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
⛔ A000380: 매출 0 제외 후 유효 데이터 2개 → 예측 제외
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
⛔ A000840: 매출 0 제외 후 유효 데이터 1개 → 예측 제외
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에 결측치가 존재 → 보간 또는 제거 필요
❌ 값에

In [55]:
results_df = forecast_df

# results_df[results_df['ticker'] == "A123860"]
revenue_df_with_forecast = pd.concat([sorted_df_resize, results_df])
revenue_growth = result_df = compute_yoy_and_ttm_growth(revenue_df_with_forecast)

In [56]:
path = r"C:\Users\MetaM\PycharmProjects\stock_forecast\Korea_Market\analysis\한군기업배출예측_단순SARIMA2.csv"

revenue_growth.to_csv(path)
revenue_growth

,date,ticker,value,forecast,yoy_growth,ttm_current,ttm_past,TTM_growth
0,2004-03-31,A000010,792089000.0,0,NaN,NaN,NaN,NaN
1,2004-06-30,A000010,761788000.0,0,NaN,NaN,NaN,NaN
2,2004-09-30,A000010,739240000.0,0,NaN,NaN,NaN,NaN
3,2004-12-31,A000010,755109000.0,0,NaN,7.620565e+08,NaN,NaN
4,2005-03-31,A000010,706788000.0,0,-0.107691,7.407312e+08,NaN,NaN
...,...,...,...,...,...,...,...,...
192429,2024-03-31,A950220,0.0,0,NaN,0.000000e+00,0.0,NaN
192430,2024-06-30,A950220,0.0,0,NaN,0.000000e+00,0.0,NaN
192431,2024-09-30,A950220,1090765.0,0,inf,2.726912e+05,0.0,inf
192432,2024-12-31,A950220,-920381.0,0,-inf,4.259600e+04,0.0,inf


In [57]:

def get_top_ttm_growth(df, n=10):
    # 날짜를 datetime으로 변환
    df['date'] = pd.to_datetime(df['date'])

    # TTM_growth가 NaN이거나 inf/-inf인 행 제거
    df_clean = df[np.isfinite(df['TTM_growth'])].dropna(subset=['TTM_growth'])

    # ticker별로 가장 최근 row 추출
    latest_df = df_clean.sort_values('date').groupby('ticker').tail(1)

    # TTM_growth 기준 내림차순 정렬 후 상위 n개 선택
    top_n = latest_df.sort_values('TTM_growth', ascending=False).head(n)

    return top_n[['ticker', 'TTM_growth']]

In [58]:
top_result = get_top_ttm_growth(revenue_df, n=500)  # 예: 상위 20개 ticker

In [59]:
top_result[top_result['TTM_growth'] >=0.328]


,ticker,TTM_growth
51850,A016970,272.439212
193847,A475830,164.263519
189771,A397030,102.560000
184797,A323990,98.842000
102833,A053170,69.628227
...,...,...
165607,A189330,0.332065
80669,A037110,0.331538
139580,A092190,0.330934
34509,A008720,0.330211


In [48]:
sorted_df_resize[(sorted_df_resize['value']!=0) & (sorted_df_resize['ticker']=='A424980') ]

,date,ticker,value,forecast
5494366,2022-06-30,A424980,16133046.96,0
5494379,2022-09-30,A424980,12823643.45,0
5494414,2023-03-31,A424980,2177196.69,0
5494437,2023-06-30,A424980,2747572.15,0
5494470,2023-09-30,A424980,2073730.38,0
5494496,2023-12-31,A424980,2371717.52,0
5494528,2024-03-31,A424980,624521.00,0
5494561,2024-06-30,A424980,2160813.28,0
5494594,2024-09-30,A424980,3109839.45,0
5494625,2024-12-31,A424980,4256298.03,0
